# Stage 5 — Expert comparison agent (video vs. textbook)

Compares the content extracted from your video frames (Stage 3's `manifest.json`) against **the specific topic in your textbooks that matches the video's title**, and produces a similarity score report.

The four steps as agreed:
1. **Parse textbooks with structure** — TOC bookmarks → topic tree (font-size fallback when no bookmarks), section text chunked with page provenance, cached so re-runs are instant
2. **Topic scoping** — the video title selects candidate sections; **you confirm** the scope before any scoring
3. **Two-direction scoring** — how much of the *video* is grounded in the section, and how much of the *section* the video covers, plus a harmonic-mean headline score
4. **Threshold calibration** — score distribution + boundary examples so "75%" becomes a cutoff you actually chose

No vector DB — with a few textbooks, cached NumPy embeddings are instant. No GPU needed (CPU runtime is fine; GPU just embeds faster).

In [ ]:
# @title 1. Setup { display-mode: "form" }
!pip install -q pymupdf sentence-transformers pandas matplotlib

import json, math, re, shutil, time, hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import fitz  # PyMuPDF
print("PyMuPDF", fitz.__doc__ or fitz.version)


In [ ]:
# @title 2. Stage 5 configuration { display-mode: "form" }

@dataclass
class Stage5Config:
    video_title: str = ""               # ← EDIT: e.g. "Working with Vector Embeddings".
                                        #   Leave "" to be prompted from the manifest/interactively.

    # --- embeddings ---
    embedding_model: str = "BAAI/bge-small-en-v1.5"   # solid default; "all-MiniLM-L6-v2" is faster

    # --- textbook chunking ---
    chunk_words: int = 350              # ~450 tokens
    chunk_overlap_words: int = 60

    # --- topic scoping ---
    auto_scope: bool = True             # True: auto-select EVERY section (across ALL books) whose title
                                        #   contains a keyword from the video title, with context checked
    scope_min_context: float = 0.45     # semantic floor for keyword hits — filters false friends
                                        #   ("graphics" won't ride in on "graph")
    scope_candidates: int = 15          # candidates shown for the manual path / override
    min_section_words: int = 100        # ignore trivially short TOC entries (title pages etc.)

    # --- scoring ---
    similarity_threshold: float = 0.75  # your "75%" — calibrate with step 4 before trusting it
    collapse_threshold: float = 0.95    # consecutive frames above this cosine are grouped
                                        # (compensates for the skipped duplicate-eliminator stage)
    exclude_frame_types: tuple = ("talking_head", "error")

    # --- paths ---
    work_dir: str = "/content/stage5"

    def __post_init__(self):
        self.books_dir  = str(Path(self.work_dir) / "books")
        self.cache_dir  = str(Path(self.work_dir) / "cache")
        self.report_dir = str(Path(self.work_dir) / "report")

CFG = Stage5Config()
for d in (CFG.work_dir, CFG.books_dir, CFG.cache_dir, CFG.report_dir):
    Path(d).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(CFG), indent=2))


## Inputs — the Stage 3 manifest and your textbook PDFs

Upload the `manifest.json` produced by Stage 3, and your PDFs (either upload them directly or point at a Drive folder).

In [ ]:
# @title Option A — upload manifest.json + PDF files from your PC
from google.colab import files

print("Select manifest.json AND your PDF textbooks (multi-select works):")
up = files.upload()
for name in up:
    dest = CFG.work_dir + "/manifest.json" if name.endswith(".json") else f"{CFG.books_dir}/{name}"
    shutil.move(name, dest)
print("Done.")


In [ ]:
# @title Option B — from Google Drive
from google.colab import drive
drive.mount("/content/drive")

MANIFEST_SRC = "/content/drive/MyDrive/pipeline/manifest.json"     # ← EDIT
BOOKS_SRC    = "/content/drive/MyDrive/pipeline/textbooks"          # ← EDIT: folder of PDFs

shutil.copy(MANIFEST_SRC, CFG.work_dir + "/manifest.json")
for p in Path(BOOKS_SRC).glob("*.pdf"):
    shutil.copy(p, Path(CFG.books_dir) / p.name)
print("Copied manifest +", len(list(Path(CFG.books_dir).glob("*.pdf"))), "PDFs")


In [ ]:
# @title 3. Load the manifest and assemble each frame's text

manifest = json.loads(Path(CFG.work_dir, "manifest.json").read_text())

def frame_text(rec: dict) -> str:
    c = rec.get("extracted_content", {}) or {}
    parts = [c.get("slide_title"), c.get("content_text"),
             c.get("code"), c.get("diagram_description"), c.get("raw_description")]
    return "\n".join(str(p) for p in parts if p)

frames = []
for rec in manifest["frames"]:
    if "extracted_content" not in rec:
        continue
    ftype = rec["extracted_content"].get("frame_type", "other")
    txt = frame_text(rec)
    if ftype in CFG.exclude_frame_types or not txt.strip():
        continue
    frames.append({"frame_id": rec["frame_id"],
                   "timestamp_hms": rec.get("timestamp_hms", "?"),
                   "timestamp_sec": rec.get("timestamp_sec", 0),
                   "frame_type": ftype, "text": txt})

print(f'{len(manifest["frames"])} frames in manifest → {len(frames)} scoreable '
      f'(excluded types: {CFG.exclude_frame_types})')

if not CFG.video_title:
    CFG.video_title = input("Video title (used to select the textbook topic): ").strip()
print("Video title:", CFG.video_title)


## Step 1 — Parse textbooks into a topic tree

TOC bookmarks first (`doc.get_toc()`); if a book has none, a font-size heuristic finds headings (lines set noticeably larger than the body text). Each section's text is the page span from its heading to the next same-or-higher-level heading, chunked with page provenance. Parsed books are cached by file hash, so this cell only pays the cost once per book.

In [ ]:
# @title 4. Textbook parser (TOC-first, font-size fallback, cached)

def _file_key(path: Path) -> str:
    return hashlib.md5(f"{path.name}:{path.stat().st_size}".encode()).hexdigest()[:12]

def _chunk_words(text: str, size: int, overlap: int):
    words = text.split()
    step = max(1, size - overlap)
    for start in range(0, max(1, len(words)), step):
        piece = words[start:start + size]
        if len(piece) < 30 and start > 0:   # tail too small to stand alone
            break
        yield " ".join(piece)

def _sections_from_toc(doc) -> list:
    toc = doc.get_toc()          # [[level, title, page], ...] 1-based pages
    if not toc:
        return []
    secs = []
    for i, (level, title, page) in enumerate(toc):
        end_page = doc.page_count
        for lvl2, _, pg2 in toc[i + 1:]:
            if lvl2 <= level:
                end_page = pg2 - 1
                break
        secs.append({"level": level, "title": title.strip(),
                     "page_start": page, "page_end": max(page, end_page)})
    return secs

def _sections_from_fonts(doc) -> list:
    """No bookmarks: treat lines with font size ≥ 1.3× the body size as headings."""
    sizes = {}
    lines = []          # (page_1based, size, text)
    for pno in range(doc.page_count):
        for block in doc.load_page(pno).get_text("dict")["blocks"]:
            for line in block.get("lines", []):
                text = "".join(s["text"] for s in line.get("spans", [])).strip()
                if not text:
                    continue
                size = round(max(s["size"] for s in line["spans"]), 1)
                sizes[size] = sizes.get(size, 0) + len(text)
                lines.append((pno + 1, size, text))
    if not lines:
        return []
    body = max(sizes, key=sizes.get)                     # most common size = body text
    heads = [(pg, sz, tx) for pg, sz, tx in lines
             if sz >= body * 1.3 and 3 < len(tx) < 120]
    secs = []
    for i, (pg, sz, tx) in enumerate(heads):
        end = heads[i + 1][0] if i + 1 < len(heads) else doc.page_count
        secs.append({"level": 1 if sz >= body * 1.6 else 2, "title": tx,
                     "page_start": pg, "page_end": max(pg, end)})
    return secs

def parse_book(pdf_path: Path, cfg) -> dict:
    cache = Path(cfg.cache_dir) / f"{_file_key(pdf_path)}.json"
    if cache.exists():
        book = json.loads(cache.read_text())
        print(f'  {pdf_path.name}: cached ({len(book["sections"])} sections)')
        return book

    doc = fitz.open(pdf_path)
    secs = _sections_from_toc(doc)
    source = "toc"
    if not secs:
        secs = _sections_from_fonts(doc)
        source = "font-heuristic"

    page_text = [doc.load_page(p).get_text("text") for p in range(doc.page_count)]
    kept = []
    for s in secs:
        text = "\n".join(page_text[s["page_start"] - 1:s["page_end"]])
        if len(text.split()) < cfg.min_section_words:
            continue
        s["chunks"] = [{"text": ch,
                        "pages": f'{s["page_start"]}–{s["page_end"]}'}
                       for ch in _chunk_words(text, cfg.chunk_words, cfg.chunk_overlap_words)]
        if s["chunks"]:
            kept.append(s)
    doc.close()

    book = {"file": pdf_path.name, "structure_source": source, "sections": kept}
    cache.write_text(json.dumps(book))
    print(f'  {pdf_path.name}: parsed via {source} → {len(kept)} usable sections, '
          f'{sum(len(s["chunks"]) for s in kept)} chunks')
    return book


BOOKS = [parse_book(p, CFG) for p in sorted(Path(CFG.books_dir).glob("*.pdf"))]
assert BOOKS, f"No PDFs found in {CFG.books_dir}"

# flat list of (book_idx, section_idx) for scoping
ALL_SECTIONS = [(bi, si) for bi, b in enumerate(BOOKS) for si in range(len(b["sections"]))]
print(f"\n{len(BOOKS)} books · {len(ALL_SECTIONS)} sections total")


In [ ]:
# @title 5. Load the embedding model (one model for EVERYTHING — titles, chunks, frames)
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(CFG.embedding_model)
print("Embedding model:", CFG.embedding_model,
      "· dim:", embedder.get_sentence_embedding_dimension())

def embed(texts: list) -> np.ndarray:
    """L2-normalized embeddings → dot product == cosine similarity."""
    return embedder.encode(texts, normalize_embeddings=True,
                           show_progress_bar=len(texts) > 50, batch_size=64)


## Step 2 — Topic scoping: every relevant section from every book

With `auto_scope=True` (default), the scope is built automatically:

1. **Keyword pass** — every section, in every book, whose title contains a keyword from the video title (exact word match with plural handling — "graph" matches "Graphs" and "Building a Graph", but not "Graphics")
2. **Context check** — keyword hits must also clear a semantic floor (`scope_min_context`) against the video title, so a coincidental word in an unrelated book is dropped
3. **Nested-section collapse** — if a chapter and its own subsections both match, only the chapter is kept, so chunks aren't counted twice

Everything that qualifies is pooled — one book contributing three sections and another contributing one is normal and correct. The cell prints exactly what was selected per book; `SELECTED_OVERRIDE` in the next cell lets you hand-pick instead if you disagree.

In [ ]:
# @title 6. Build the scope — keyword + context auto-selection across ALL books

STOPWORDS = {"a","an","the","and","or","of","in","on","for","with","to","is","are",
             "how","what","why","using","introduction","tutorial","course","part",
             "chapter","lesson","video","complete","guide","beginners"}

def title_keywords(title: str) -> list:
    words = re.findall(r"[a-zA-Z]{3,}", title.lower())
    return [w for w in words if w not in STOPWORDS]

def keyword_in(kw: str, text: str) -> bool:
    """Exact word match with simple plural handling — no prefix matching."""
    for tok in re.findall(r"[a-zA-Z]+", text.lower()):
        if tok == kw or tok == kw + "s" or tok == kw + "es" or kw == tok + "s":
            return True
    return False

def section_label(bi, si):
    b, s = BOOKS[bi], BOOKS[bi]["sections"][si]
    return f'{s["title"]} — {b["file"]} (pp. {s["page_start"]}–{s["page_end"]}, level {s["level"]})'

KEYWORDS = title_keywords(CFG.video_title)
print(f'Video title: "{CFG.video_title}" → keywords: {KEYWORDS}\n')

# semantic score for every section (used as context check AND for the manual/override list)
section_title_texts = [f'{Path(BOOKS[bi]["file"]).stem}: {BOOKS[bi]["sections"][si]["title"]}'
                       for bi, si in ALL_SECTIONS]
title_vec = embed([CFG.video_title])[0]
sem_scores = embed(section_title_texts) @ title_vec

# ---- pass 1+2: keyword hit AND context floor ----
hits = []
for idx, (bi, si) in enumerate(ALL_SECTIONS):
    s = BOOKS[bi]["sections"][si]
    if any(keyword_in(kw, s["title"]) for kw in KEYWORDS) \
            and sem_scores[idx] >= CFG.scope_min_context:
        hits.append((bi, si))

# ---- pass 3: drop sections nested inside an already-selected section of the same book ----
def _contains(a, b):   # a contains b (same book)
    return a["page_start"] <= b["page_start"] and a["page_end"] >= b["page_end"] \
           and (a["page_start"], a["page_end"]) != (b["page_start"], b["page_end"])

AUTO_SELECTED = []
for bi, si in hits:
    s = BOOKS[bi]["sections"][si]
    if any(b2 == bi and _contains(BOOKS[b2]["sections"][s2], s) for b2, s2 in hits):
        continue        # a matching parent already covers these pages
    AUTO_SELECTED.append((bi, si))

if CFG.auto_scope and AUTO_SELECTED:
    per_book = {}
    for bi, si in AUTO_SELECTED:
        per_book.setdefault(BOOKS[bi]["file"], []).append(si)
    print(f"Auto-selected {len(AUTO_SELECTED)} sections across {len(per_book)} book(s):")
    for bi, si in AUTO_SELECTED:
        idx = ALL_SECTIONS.index((bi, si))
        print(f"  ✓ {sem_scores[idx]:.3f}  {section_label(bi, si)}")
    books_no_hit = [b["file"] for b in BOOKS if b["file"] not in per_book]
    if books_no_hit:
        print("\nNo qualifying section in:", ", ".join(books_no_hit),
              "\n(no title keyword match above the context floor — lower scope_min_context "
              "or use SELECTED_OVERRIDE if that's wrong)")
else:
    if CFG.auto_scope:
        print("⚠️ No section title matched the keywords — falling back to semantic ranking. "
              "Pick indices with SELECTED_OVERRIDE in the next cell.")
    order = np.argsort(-sem_scores)[:CFG.scope_candidates]
    print("\nRanked candidates:")
    for rank, idx in enumerate(order):
        bi, si = ALL_SECTIONS[idx]
        print(f"  [{rank}] {sem_scores[idx]:.3f}  {section_label(bi, si)}")
    CANDIDATES = [ALL_SECTIONS[i] for i in order]


In [ ]:
# @title 7. Finalize the scope (override only if you disagree with the auto-selection)
SELECTED_OVERRIDE = None    # e.g. [0, 2] to hand-pick from the ranked candidate list instead

if SELECTED_OVERRIDE is not None:
    scope = [CANDIDATES[i] for i in SELECTED_OVERRIDE]
elif CFG.auto_scope and AUTO_SELECTED:
    scope = AUTO_SELECTED
else:
    raise ValueError("No auto-selection available — set SELECTED_OVERRIDE from the candidate list above.")

scope_chunks, scope_meta = [], []
for bi, si in scope:
    s = BOOKS[bi]["sections"][si]
    for ci, ch in enumerate(s["chunks"]):
        scope_chunks.append(ch["text"])
        scope_meta.append({"book": BOOKS[bi]["file"], "section": s["title"],
                           "pages": ch["pages"], "chunk_id": f"{bi}.{si}.{ci}"})

print("Scoring scope:")
for bi, si in scope:
    print("  •", section_label(bi, si))
print(f"{len(scope_chunks)} textbook chunks in scope "
      f"from {len(set(m['book'] for m in scope_meta))} book(s)")


## Step 3 — Score the video against the scoped section

Frame texts and textbook chunks are embedded with the same model. Consecutive frames whose embeddings exceed `collapse_threshold` cosine are **grouped** first — the dedup you skipped, done for free at scoring time, so repeated slides can't inflate coverage. Then:

- **Video groundedness** — fraction of frame groups whose best chunk match ≥ threshold ("how much of the video is in the book")
- **Topic coverage** — fraction of in-scope chunks reached by some group at ≥ threshold ("how much of the book's topic the video covers")
- **Headline score** — harmonic mean of the two

In [ ]:
# @title 8. Scoring

frame_vecs = embed([f["text"] for f in frames])
chunk_vecs = embed(scope_chunks)

# ---- collapse consecutive near-identical frames into groups ----
groups, current = [], [0]
for i in range(1, len(frames)):
    if float(frame_vecs[i] @ frame_vecs[current[-1]]) >= CFG.collapse_threshold:
        current.append(i)
    else:
        groups.append(current); current = [i]
groups.append(current)
group_vecs = np.stack([frame_vecs[g].mean(axis=0) for g in groups])
group_vecs /= np.linalg.norm(group_vecs, axis=1, keepdims=True)
print(f"{len(frames)} frames collapsed into {len(groups)} distinct content groups")

# ---- similarity matrix: groups × chunks ----
S = group_vecs @ chunk_vecs.T                     # cosine, thanks to normalization
best_chunk = S.argmax(axis=1)
best_score = S.max(axis=1)

thr = CFG.similarity_threshold
video_groundedness = float((best_score >= thr).mean())
chunk_best = S.max(axis=0)
topic_coverage = float((chunk_best >= thr).mean())
headline = (0.0 if video_groundedness + topic_coverage == 0 else
            2 * video_groundedness * topic_coverage / (video_groundedness + topic_coverage))

print(f"\nAt threshold {thr}:")
print(f"  Video groundedness : {video_groundedness:6.1%}  (groups matched in the book)")
print(f"  Topic coverage     : {topic_coverage:6.1%}  (book chunks the video reaches)")
print(f"  Headline score     : {headline:6.1%}  (harmonic mean)")

# ---- per-frame table ----
rows = []
for gi, g in enumerate(groups):
    for fi in g:
        f, m = frames[fi], scope_meta[best_chunk[gi]]
        rows.append({"frame_id": f["frame_id"], "time": f["timestamp_hms"],
                     "type": f["frame_type"], "group": gi,
                     "score": round(float(best_score[gi]), 3),
                     "matched": best_score[gi] >= thr,
                     "best_section": m["section"], "pages": m["pages"],
                     "book": m["book"]})
df = pd.DataFrame(rows).sort_values("time").reset_index(drop=True)
df


## Step 4 — Calibrate the threshold

Cosine similarity is not "percent similar." Between OCR'd slide language and formal textbook prose, ~0.60–0.70 is usually already a genuine topical match. Look at the distribution and the boundary pairs below, then set `similarity_threshold` in the config to the value where matches *you'd* call similar start — and re-run the scoring cell.

In [ ]:
# @title 9. Score distribution + boundary examples
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(best_score, bins=24, color="#5DCAA5", edgecolor="white")
ax.axvline(thr, color="#D85A30", linestyle="--", label=f"threshold = {thr}")
ax.set_xlabel("best cosine similarity per content group"); ax.set_ylabel("groups")
ax.legend(); plt.tight_layout(); plt.show()

order = np.argsort(np.abs(best_score - thr))[:4]     # pairs nearest the boundary
for gi in order:
    fi = groups[gi][0]
    print("═" * 72)
    print(f'score {best_score[gi]:.3f}  ({"MATCH" if best_score[gi] >= thr else "below"})  '
          f'frame {frames[fi]["frame_id"]} @ {frames[fi]["timestamp_hms"]}')
    print("FRAME  :", frames[fi]["text"][:220].replace("\n", " "), "…")
    print("CHUNK  :", scope_chunks[best_chunk[gi]][:220].replace("\n", " "), "…")
    m = scope_meta[best_chunk[gi]]
    print("SOURCE :", m["book"], "·", m["section"], "· pp.", m["pages"])


In [ ]:
# @title 10. Save the report (JSON + CSV) and zip it

report = {
    "video_title": CFG.video_title,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "embedding_model": CFG.embedding_model,
    "scope": [section_label(bi, si) for bi, si in scope],
    "threshold": CFG.similarity_threshold,
    "results": {"video_groundedness": round(video_groundedness, 4),
                "topic_coverage": round(topic_coverage, 4),
                "headline_score": round(headline, 4),
                "frames_scored": len(frames),
                "content_groups": len(groups),
                "chunks_in_scope": len(scope_chunks)},
    "frames": df.to_dict(orient="records"),
}
Path(CFG.report_dir, "stage5_report.json").write_text(json.dumps(report, indent=2))
df.to_csv(Path(CFG.report_dir, "stage5_frames.csv"), index=False)

zip_path = shutil.make_archive(str(Path(CFG.work_dir) / "stage5_output"), "zip", CFG.report_dir)
print("Saved:", zip_path)
# from google.colab import files; files.download(zip_path)


## Reading the report

`stage5_report.json` carries the headline numbers plus every frame's best match with book, section, and page range — so a low-coverage result is immediately actionable ("the video never touches pp. 214–221 on normalization"). If you later want the timeline visual — video minutes on one axis, matched textbook pages highlighted — that's a natural next cell to add on top of `stage5_frames.csv`.